In [ ]:
# Source-only RA ablation | Kaggle job: 2a_22ch_4cls_ra_on
# Settings: Accelerator = GPU T4, Internet = On. Run with "Save Version -> Save & Run All".
import codecs, os, select, shutil, subprocess, sys, time
from pathlib import Path

BRANCH = 'feature/full-mamba-source-only-ra-ablation'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
RUN_TAG = '2a_22ch_4cls_ra_on'
DATASET = 'bcic2a'
PREPROCESSING = {'riemannian_alignment': True}
TARGET_SUBJECTS = list(range(1, 10))
EXCLUDED = set()
SEEDS = [0]   # same seed as the existing source-only results; add 1, 2 for repeated seeds

REPO_PATH = Path('/kaggle/working/repo')
MNE_DATA = Path('/tmp/mne_data')            # MOABB downloads (kept out of the output)
OUTPUT = Path(f'/kaggle/working/ra_ablation_{RUN_TAG}')
OUTPUT.mkdir(parents=True, exist_ok=True)


def run(command, cwd=None, env=None, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                text = decoder.decode(chunk)
                print(text, end='', flush=True); log.write(text); log.flush()
            elif process.poll() is None:
                beat = f'[heartbeat] elapsed={(time.monotonic() - started) / 60:.1f}m\n'
                print(beat, end='', flush=True); log.write(beat); log.flush()
            else:
                break
    if process.wait() != 0:
        raise RuntimeError(f'Command failed. Log: {log_path}')


# ---- Setup ----
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1',
     '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run(['nvidia-smi'])

environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
                    'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA),
                    'MNE_DATASETS_BNCI_PATH': str(MNE_DATA),
                    'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA)})
MNE_DATA.mkdir(parents=True, exist_ok=True)

# ---- One process per fold; results are archived right after each fold ----
for seed in SEEDS:
    for subject in TARGET_SUBJECTS:
        if subject in EXCLUDED:
            print(f'Skip S{subject:03d}: excluded recording.', flush=True)
            continue
        fold = f'seed{seed}_S{subject:03d}'
        if (OUTPUT / f'{fold}_results.zip').exists():
            print(f'Skip {fold}: already finished.', flush=True)
            continue
        override = (
            "import yaml; from pathlib import Path; "
            "p=Path('configs/full_mamba_source_only.yaml'); c=yaml.safe_load(p.read_text()); "
            f"c['subject_ids']=[{subject}]; c['log_every_n_epochs']=10; c['enable_progress_bar']=False; "
            f"c['preprocessing'][{DATASET!r}].update({PREPROCESSING!r}); "
            "p.write_text(yaml.safe_dump(c, sort_keys=False))"
        )
        run(['git', 'checkout', '--', 'configs/full_mamba_source_only.yaml'], cwd=REPO_PATH)
        run([PYTHON, '-c', override], cwd=REPO_PATH)
        results_dir = REPO_PATH / 'results'
        shutil.rmtree(results_dir, ignore_errors=True)
        print(f'===== {RUN_TAG} | seed {seed} | target S{subject:03d} | {PREPROCESSING} =====', flush=True)
        run([PYTHON, '-u', 'train_pipeline.py', '--model', 'full_mamba_source_only',
             '--dataset', DATASET, '--loso', '--gpu_id', '0', '--seed', seed],
            cwd=REPO_PATH, env=environment, log_path=OUTPUT / f'{fold}.log')
        shutil.make_archive(str(OUTPUT / f'{fold}_results'), 'zip',
                            root_dir=REPO_PATH, base_dir='results')
        print(f'Saved {OUTPUT / f"{fold}_results.zip"}', flush=True)

# ---- Summary of this job ----
import json, zipfile
rows = []
for archive in sorted(OUTPUT.glob('seed*_S*_results.zip')):
    seed = int(archive.name.split('_')[0][4:])
    with zipfile.ZipFile(archive) as zf:
        for name in zf.namelist():
            if '/subjects/subject_' in name and name.endswith('.json'):
                rows.append({**json.loads(zf.read(name)), 'seed': seed})
with open(OUTPUT / 'summary.csv', 'w') as f:
    f.write('run,seed,subject_id,test_acc,test_kappa,best_val_acc,train_time_min\n')
    for r in sorted(rows, key=lambda r: (r['seed'], r['subject_id'])):
        f.write(f"{RUN_TAG},{r['seed']},{r['subject_id']},{r['test_acc']:.4f},{r['test_kappa']:.4f},"
                f"{r['best_val_acc']:.4f},{r['train_time_min']:.1f}\n")
print(open(OUTPUT / 'summary.csv').read())
